# Notebook 02 — Fusion des tables & construction de la base ML
**Projet : Gestion des ventes et prévision des sorties de stock**

Ce notebook prend en entrée le fichier Excel nettoyé (`stock_boutique_nettoye.xlsx`)
et produit la base d'apprentissage (`base_apprentissage_ml.csv`) utilisée pour la
modélisation en régression supervisée.

### Stratégie de fusion
```
Inventaire ──(left join sur 'reference')──► Sorties  ← table centrale
                                                │
                  Entrées ──(groupby SUM)──► left join
                                                │
                                                ▼
                                   base_apprentissage_ml.csv
```

### Variables produites
| Colonne | Rôle | Source |
|---|---|---|
| `quantite_vendue` | **Variable cible Y** | Sorties |
| `prix_unitaire` | Variable explicative X | Sorties |
| `categorie` | Variable explicative X | Sorties |
| `quantite_entree` | Variable explicative X | Entrées (agrégé) |
| `stock_actuel` | Variable explicative X | Inventaire |
| `seuil_d_alerte` | Variable explicative X | Inventaire |

---
## 0. Imports et configuration

In [1]:
import pandas as pd
import numpy as np

# Options d'affichage : toutes les colonnes visibles, nombres formatés avec séparateur
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Chemin du fichier Excel issu du notebook 01 (nettoyage)
FICHIER = 'stock_boutique_nettoye.xlsx'

print('Librairies chargées ✓')
print(f'Fichier source : {FICHIER}')

Librairies chargées ✓
Fichier source : stock_boutique_nettoye.xlsx


---
## 1. Chargement des quatre feuilles Excel

Le fichier nettoyé contient quatre feuilles indépendantes :

| Feuille | Lignes | Colonnes | Rôle dans la fusion |
|---|---|---|---|
| `Marchandises` | 500 | 8 | Catalogue produits (jointure optionnelle) |
| `Entrees` | 800 | 8 | Mouvements d'approvisionnement |
| `Sorties` | 900 | 8 | **Table centrale** — une ligne = une vente |
| `Inventaire` | 500 | 10 | Stock final et seuil d'alerte par produit |

In [2]:
# Chargement de chaque feuille dans un DataFrame distinct
df_marchandises = pd.read_excel(FICHIER, sheet_name='Marchandises')
df_entrees      = pd.read_excel(FICHIER, sheet_name='Entrees')
df_sorties      = pd.read_excel(FICHIER, sheet_name='Sorties')
df_inventaire   = pd.read_excel(FICHIER, sheet_name='Inventaire')

# Vérification des dimensions après chargement
print('Dimensions des tables chargées :')
print(f'  Marchandises : {df_marchandises.shape[0]:>4} lignes × {df_marchandises.shape[1]} colonnes')
print(f'  Entrees      : {df_entrees.shape[0]:>4} lignes × {df_entrees.shape[1]} colonnes')
print(f'  Sorties      : {df_sorties.shape[0]:>4} lignes × {df_sorties.shape[1]} colonnes')
print(f'  Inventaire   : {df_inventaire.shape[0]:>4} lignes × {df_inventaire.shape[1]} colonnes')

Dimensions des tables chargées :
  Marchandises :  500 lignes × 8 colonnes
  Entrees      :  800 lignes × 8 colonnes
  Sorties      :  900 lignes × 8 colonnes
  Inventaire   :  500 lignes × 10 colonnes


In [3]:
# Aperçu des colonnes et des 3 premières lignes de chaque table
tables = [
    ('Marchandises', df_marchandises),
    ('Entrees',      df_entrees),
    ('Sorties',      df_sorties),
    ('Inventaire',   df_inventaire),
]

for nom, df in tables:
    print(f'\n{"─" * 60}')
    print(f'  TABLE : {nom}')
    print(f'  Colonnes : {list(df.columns)}')
    display(df.head(3))


────────────────────────────────────────────────────────────
  TABLE : Marchandises
  Colonnes : ['reference', 'designation', 'categorie', 'seuil_d_alerte', 'unite', 'stock_initial', 'prix_unitaire', 'total']


,reference,designation,categorie,seuil_d_alerte,unite,stock_initial,prix_unitaire,total
0,SMP-0001,Smartphone Edge 001,Smartphone,9,Piece,235,"993,150.83","233,390,445.05"
1,ELM-0001,Lave-linge 001,Electromenager,18,Piece,44,"891,046.06","39,206,026.64"
2,SMP-0002,Smartphone Edge 002,Smartphone,16,Piece,145,"1,579,871.45","229,081,360.25"



────────────────────────────────────────────────────────────
  TABLE : Entrees
  Colonnes : ['date', 'reference', 'designation', 'categorie', 'prix_unitaire', 'quantite', 'total', 'type']


,date,reference,designation,categorie,prix_unitaire,quantite,total,type
0,2025-01-01,GEN-0049,Groupe 8kVA 049,Groupe electrogene,"3,694,748.41",70,"258,632,388.70",Entree
1,2025-01-02,ELM-0095,Lave-linge 095,Electromenager,"81,234.89",109,"8,854,603.01",Entree
2,2025-01-02,ELM-0093,Climatiseur 093,Electromenager,"873,960.60",11,"9,613,566.60",Entree



────────────────────────────────────────────────────────────
  TABLE : Sorties
  Colonnes : ['date', 'reference', 'designation', 'categorie', 'prix_unitaire', 'quantite', 'total', 'type']


,date,reference,designation,categorie,prix_unitaire,quantite,total,type
0,2025-01-01,MUS-0094,Piano numerique 094,Instrument de musique,"1,456,997.89",37,"53,908,921.93",Sortie
1,2025-01-02,ELM-0071,Lave-linge 071,Electromenager,"272,779.92",59,"16,094,015.28",Sortie
2,2025-01-02,SMP-0065,Smartphone Ultra 065,Smartphone,"307,551.03",1,"307,551.03",Sortie



────────────────────────────────────────────────────────────
  TABLE : Inventaire
  Colonnes : ['reference', 'designation', 'categorie', 'seuil_d_alerte', 'stock_initial', 'entrees', 'sorties', 'stock_final', 'valeur', 'statut']


,reference,designation,categorie,seuil_d_alerte,stock_initial,entrees,sorties,stock_final,valeur,statut
0,SMP-0001,Smartphone Edge 001,Smartphone,9,235,83,87,231,"229,417,841.73",stock normal
1,ELM-0001,Lave-linge 001,Electromenager,18,44,2,0,46,"40,988,118.76",stock normal
2,SMP-0002,Smartphone Edge 002,Smartphone,16,145,72,32,185,"292,276,218.25",stock normal


---
## 2. Étape A — Jointure Inventaire → Sorties

### Objectif
Ajouter à chaque ligne de vente (Sorties) les informations de stock correspondantes :
- `stock_actuel` : quantité disponible en rayon après tous les mouvements (= `stock_final`)
- `seuil_d_alerte` : seuil en dessous duquel le produit doit être réapprovisionné

### Type de jointure
**Left join** sur `reference` : toutes les 900 lignes de Sorties sont conservées.
Les produits absents de l'Inventaire recevront des `NaN` (traités à l'étape 6).

In [4]:
# Sélection des colonnes utiles depuis l'Inventaire
# On garde uniquement les colonnes qui seront utilisées comme variables ML
# 'statut' est exclu : c'est une variable texte dérivée de stock_actuel et seuil_d_alerte
# la conserver dans df_base ne ferait que polluer la base intermédiaire
cols_inventaire = ['reference', 'seuil_d_alerte', 'stock_final']
df_inv_light = (
    df_inventaire[cols_inventaire]
    .rename(columns={'stock_final': 'stock_actuel'})
    .copy()
)

print('Colonnes extraites de l\'Inventaire :', list(df_inv_light.columns))
print(f'Plage de stock_actuel : [{df_inv_light["stock_actuel"].min()}, {df_inv_light["stock_actuel"].max()}]')
print(f'  → {(df_inv_light["stock_actuel"] < 0).sum()} produits en stock négatif (anomalie métier identifiée en notebook 01)')
display(df_inv_light.head(3))

Colonnes extraites de l'Inventaire : ['reference', 'seuil_d_alerte', 'stock_actuel']
Plage de stock_actuel : [0, 712]
  → 0 produits en stock négatif (anomalie métier identifiée en notebook 01)


,reference,seuil_d_alerte,stock_actuel
0,SMP-0001,9,231
1,ELM-0001,18,46
2,SMP-0002,16,185


In [5]:
# Jointure gauche : Sorties (900 lignes) ← Inventaire (500 produits)
# Chaque ligne de vente reçoit le stock_actuel et le seuil_d_alerte de son produit
df_base = df_sorties.merge(
    df_inv_light,
    on='reference',
    how='left'
)

print(f'Sorties avant jointure  : {df_sorties.shape}')
print(f'Base après jointure     : {df_base.shape}')
print('Colonnes ajoutées       : stock_actuel, seuil_d_alerte')
print(f'Valeurs manquantes      : {df_base[["stock_actuel","seuil_d_alerte"]].isna().sum().to_dict()}')
display(df_base.head(3))

Sorties avant jointure  : (900, 8)
Base après jointure     : (900, 10)
Colonnes ajoutées       : stock_actuel, seuil_d_alerte
Valeurs manquantes      : {'stock_actuel': 0, 'seuil_d_alerte': 0}


,date,reference,designation,categorie,prix_unitaire,quantite,total,type,seuil_d_alerte,stock_actuel
0,2025-01-01,MUS-0094,Piano numerique 094,Instrument de musique,"1,456,997.89",37,"53,908,921.93",Sortie,13,355
1,2025-01-02,ELM-0071,Lave-linge 071,Electromenager,"272,779.92",59,"16,094,015.28",Sortie,15,105
2,2025-01-02,SMP-0065,Smartphone Ultra 065,Smartphone,"307,551.03",1,"307,551.03",Sortie,8,482


---
## 3. Étape B — Agrégation des Entrées par produit

### Pourquoi agréger avant de joindre ?
La table Entrées contient **plusieurs lignes par produit** (réapprovisionnements successifs).
Si on la joint directement sur les Sorties, chaque vente serait **dupliquée** autant de fois
qu'il y a d'entrées pour ce produit — ce qui fausserait toute l'analyse.

**Solution :** calculer d'abord `quantite_entree` = somme totale des entrées par produit,
puis joindre ce résultat agrégé (une seule ligne par référence).

In [6]:
# Agrégation : somme des quantités entrées pour chaque référence produit
# Résultat : une ligne par référence avec la quantité totale approvisionnée
df_entrees_agg = (
    df_entrees
    .groupby('reference', as_index=False)
    .agg(quantite_entree=('quantite', 'sum'))
)

print(f'Entrées brutes   : {df_entrees.shape[0]:>4} lignes (plusieurs entrées par produit possible)')
print(f'Entrées agrégées : {df_entrees_agg.shape[0]:>4} lignes (exactement 1 ligne par référence)')
print(f'Produits avec au moins une entrée : {df_entrees_agg.shape[0]} / {df_marchandises.shape[0]}')
display(df_entrees_agg.head(5))

Entrées brutes   :  800 lignes (plusieurs entrées par produit possible)
Entrées agrégées :  406 lignes (exactement 1 ligne par référence)
Produits avec au moins une entrée : 406 / 500


,reference,quantite_entree
0,ELM-0001,2
1,ELM-0002,85
2,ELM-0003,128
3,ELM-0004,22
4,ELM-0005,55


---
## 4. Étape C — Jointure des Entrées agrégées

On enrichit la base (Sorties + Inventaire) avec `quantite_entree`.
Même principe : **left join** sur `reference` pour conserver toutes les ventes.
Les produits sans aucune entrée enregistrée recevront `NaN` → imputé à 0 à l'étape 6.

In [7]:
# Jointure gauche : base existante ← Entrées agrégées
# Les produits sans entrée enregistrée auront quantite_entree = NaN
df_base = df_base.merge(
    df_entrees_agg,
    on='reference',
    how='left'
)

# Suppression des colonnes parasites issues de la table Sorties
# 'total' = prix_unitaire × quantite (redondant, non utilisé comme variable ML)
# 'type'  = toujours 'Sortie' pour toutes les lignes (aucune information)
cols_a_supprimer = ['total', 'type']
df_base.drop(columns=cols_a_supprimer, inplace=True)

print(f'Base après fusion avec Entrées : {df_base.shape}')
print(f'Colonnes disponibles : {list(df_base.columns)}')
print(f'NaN dans quantite_entree : {df_base["quantite_entree"].isna().sum()} lignes (produits sans approvisionnement enregistré)')

Base après fusion avec Entrées : (900, 9)
Colonnes disponibles : ['date', 'reference', 'designation', 'categorie', 'prix_unitaire', 'quantite', 'seuil_d_alerte', 'stock_actuel', 'quantite_entree']
NaN dans quantite_entree : 173 lignes (produits sans approvisionnement enregistré)


---
## 5. Étape D — Jointure optionnelle avec Marchandises

Les colonnes `prix_unitaire` et `categorie` sont **déjà présentes** dans la table Sorties.
Cette jointure n'est utile que si des valeurs de `seuil_d_alerte` manquent dans l'Inventaire.

On vérifie le taux de complétion avant de décider.

In [8]:
# Vérification : seuil_d_alerte est-il complet après la jointure Inventaire ?
pct_manquant = df_base['seuil_d_alerte'].isna().mean() * 100
print(f'seuil_d_alerte manquant : {pct_manquant:.1f}% des lignes')

if pct_manquant > 0:
    # Activation de la jointure optionnelle pour récupérer les seuils manquants
    print('→ Jointure Marchandises activée pour combler les valeurs manquantes')
    cols_march = ['reference', 'seuil_d_alerte']
    df_march_light = df_marchandises[cols_march].rename(
        columns={'seuil_d_alerte': 'seuil_d_alerte_march'}
    )
    df_base = df_base.merge(df_march_light, on='reference', how='left')

    # Remplacement uniquement là où seuil_d_alerte est absent
    df_base['seuil_d_alerte'] = df_base['seuil_d_alerte'].fillna(
        df_base['seuil_d_alerte_march']
    )
    df_base.drop(columns=['seuil_d_alerte_march'], inplace=True)
    print(f'Jointure Marchandises appliquée ✓ — NaN restants : {df_base["seuil_d_alerte"].isna().sum()}')
else:
    # seuil_d_alerte complet depuis l'Inventaire, jointure inutile
    print('→ seuil_d_alerte complet — jointure Marchandises non nécessaire ✓')

seuil_d_alerte manquant : 0.0% des lignes
→ seuil_d_alerte complet — jointure Marchandises non nécessaire ✓


---
## 6. Traitement des valeurs manquantes

Après toutes les jointures, un seul type de valeur manquante subsiste :

| Colonne | Cause | Stratégie |
|---|---|---|
| `quantite_entree` | Produit jamais réapprovisionné dans la période | Imputation à **0** (logique métier) |
| `stock_actuel` | Produit absent de l'Inventaire | Médiane de la **catégorie** (fallback : médiane globale) |
| `seuil_d_alerte` | Idem | Médiane de la **catégorie** (fallback : médiane globale) |

In [9]:
# ── Diagnostic avant traitement ────────────────────────────────────────────
print('=== Valeurs manquantes avant imputation ===')
manquants = df_base.isna().sum()
if manquants.sum() == 0:
    print('Aucune valeur manquante détectée.')
else:
    display(manquants[manquants > 0].to_frame('nb_manquants'))

# ── Imputation 1 : quantite_entree → 0 ─────────────────────────────────────
# Un NaN signifie qu'il n'y a eu aucune entrée enregistrée pour ce produit
# La valeur logique est donc 0 (pas d'approvisionnement sur la période)
# Conversion en int : les quantités sont des entiers
df_base['quantite_entree'] = df_base['quantite_entree'].fillna(0).astype(int)

# ── Imputation 2 : stock_actuel et seuil_d_alerte → médiane par catégorie ───
# Stratégie : utiliser la médiane des produits de la même catégorie
# pour préserver la cohérence métier (les seuils varient selon le type de produit)
for col in ['stock_actuel', 'seuil_d_alerte']:
    mediane_cat = df_base.groupby('categorie')[col].transform('median')
    df_base[col] = df_base[col].fillna(mediane_cat)
    # Fallback global si toute la catégorie est vide
    df_base[col] = df_base[col].fillna(df_base[col].median())

# ── Vérification finale ─────────────────────────────────────────────────────
print('\n=== Valeurs manquantes après imputation ===')
remaining = df_base.isna().sum().sum()
print(f'Total valeurs manquantes restantes : {remaining}')

print('\n=== Types des colonnes imputées ===')
print(df_base[['quantite_entree', 'stock_actuel', 'seuil_d_alerte']].dtypes)

=== Valeurs manquantes avant imputation ===


,nb_manquants
quantite_entree,173



=== Valeurs manquantes après imputation ===
Total valeurs manquantes restantes : 0

=== Types des colonnes imputées ===
quantite_entree    int64
stock_actuel       int64
seuil_d_alerte     int64
dtype: object


---
## 7. Construction de la base d'apprentissage finale

### Choix des variables
- **Variable cible Y** : `quantite_vendue` — ce que le modèle doit prédire
- **Variables explicatives X** :
  - `prix_unitaire` : influence directe sur la demande (relation inverse attendue)
  - `categorie` : le comportement d'achat varie fortement selon le type de produit
  - `quantite_entree` : proxy de la disponibilité et de la demande anticipée
  - `stock_actuel` : niveau de stock disponible au moment de la vente
  - `seuil_d_alerte` : indicateur de tension sur le stock

> **Note :** La colonne `statut` est exclue — elle est redondante avec `stock_actuel`
> et `seuil_d_alerte` dont elle dérive, et ne ferait qu'introduire de la multicolinéarité.

In [10]:
# ── Renommage de la variable cible ─────────────────────────────────────────
# La colonne 'quantite' de la table Sorties représente les unités vendues
# On la renomme explicitement pour éviter toute ambiguité dans la suite
df_base = df_base.rename(columns={'quantite': 'quantite_vendue'})

# ── Définition des variables ────────────────────────────────────────────────
VARIABLES_X = [
    'prix_unitaire',   # Prix de vente unitaire du produit (depuis Sorties)
    'categorie',       # Catégorie du produit — à encoder avant modélisation
    'quantite_entree', # Total des entrées en stock sur la période (depuis Entrées)
    'stock_actuel',    # Stock disponible réel = stock_final (depuis Inventaire)
    'seuil_d_alerte',  # Quantité minimale avant réapprovisionnement (depuis Inventaire)
]
VARIABLE_Y = 'quantite_vendue'

# ── Sélection des colonnes finales ──────────────────────────────────────────
# On conserve les identifiants (date, reference, designation) pour la traçabilité
# mais ils ne seront pas utilisés comme variables d'entrée du modèle
cols_finales = ['date', 'reference', 'designation'] + VARIABLES_X + [VARIABLE_Y]
df_ml = df_base[cols_finales].copy()

# ── Résumé ─────────────────────────────────────────────────────────────────
print(f'Base d\'apprentissage finale : {df_ml.shape[0]} lignes × {df_ml.shape[1]} colonnes')
print(f'Variables explicatives (X)  : {VARIABLES_X}')
print(f'Variable cible         (Y)  : {VARIABLE_Y}')
print()
print('Types des colonnes :')
print(df_ml.dtypes)
print()
display(df_ml.head(5))

Base d'apprentissage finale : 900 lignes × 9 colonnes
Variables explicatives (X)  : ['prix_unitaire', 'categorie', 'quantite_entree', 'stock_actuel', 'seuil_d_alerte']
Variable cible         (Y)  : quantite_vendue

Types des colonnes :
date               datetime64[ns]
reference                  object
designation                object
prix_unitaire             float64
categorie                  object
quantite_entree             int64
stock_actuel                int64
seuil_d_alerte              int64
quantite_vendue             int64
dtype: object



,date,reference,designation,prix_unitaire,categorie,quantite_entree,stock_actuel,seuil_d_alerte,quantite_vendue
0,2025-01-01,MUS-0094,Piano numerique 094,"1,456,997.89",Instrument de musique,56,355,13,37
1,2025-01-02,ELM-0071,Lave-linge 071,"272,779.92",Electromenager,150,105,15,59
2,2025-01-02,SMP-0065,Smartphone Ultra 065,"307,551.03",Smartphone,195,482,8,1
3,2025-01-02,SMP-0047,Smartphone Lite 047,"328,380.93",Smartphone,29,0,18,17
4,2025-01-02,MUS-0009,Clavier arrangeur 009,"2,081,845.57",Instrument de musique,389,493,18,23


---
## 8. Récapitulatif de la fusion

In [11]:
# Bilan complet du pipeline de fusion
print("""
╔══════════════════════════════════════════════════════════════════╗
║         RÉCAPITULATIF DU PIPELINE DE FUSION                      ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  ÉTAPE A — Inventaire → Sorties (left join sur 'reference')      ║
║    stock_actuel    = stock_final (stock disponible réel)         ║
║    seuil_d_alerte = seuil de réapprovisionnement                 ║
║    Résultat : 900 lignes × 10 colonnes                           ║
║                                                                  ║
║  ÉTAPE B — Entrées : agrégation préalable                        ║
║    quantite_entree = SUM(quantite) GROUP BY reference            ║
║    Évite la duplication des lignes de ventes                     ║
║                                                                  ║
║  ÉTAPE C — Base ← Entrées agrégées (left join sur 'reference')   ║
║    Colonnes parasites supprimées : total, type                   ║
║    NaN → produit sans entrée enregistrée (imputé à 0)            ║
║    Résultat : 900 lignes × 9 colonnes intermédiaires             ║
║                                                                  ║
║  ÉTAPE D — Marchandises (optionnelle, activée si NaN)            ║
║    Complète seuil_d_alerte si absent de l'Inventaire             ║
║                                                                  ║
║  BASE FINALE                                                     ║
║    Y  = quantite_vendue  (variable à prédire)                    ║
║    X  = prix_unitaire · categorie · quantite_entree              ║
║         stock_actuel · seuil_d_alerte                            ║
║    Taille : 900 lignes × 9 colonnes                              ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

# Statistiques descriptives des variables numériques uniquement
cols_numeriques = ['prix_unitaire', 'quantite_entree', 'stock_actuel', 'seuil_d_alerte', 'quantite_vendue']
print('Statistiques descriptives des variables numériques :')
display(df_ml[cols_numeriques].describe().round(2))

# Signalement de l'anomalie stock_actuel négatif
nb_negatifs = (df_ml['stock_actuel'] < 0).sum()
print(f'\n⚠ stock_actuel < 0 : {nb_negatifs} lignes ({nb_negatifs/len(df_ml)*100:.1f}%)')
print('  → Ruptures de stock confirmées (anomalie métier déjà identifiée en notebook 01)')
print('  → À gérer lors de la modélisation : clip(lower=0) ou conservation selon le choix métier')


╔══════════════════════════════════════════════════════════════════╗
║         RÉCAPITULATIF DU PIPELINE DE FUSION                      ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  ÉTAPE A — Inventaire → Sorties (left join sur 'reference')      ║
║    stock_actuel    = stock_final (stock disponible réel)         ║
║    seuil_d_alerte = seuil de réapprovisionnement                 ║
║    Résultat : 900 lignes × 10 colonnes                           ║
║                                                                  ║
║  ÉTAPE B — Entrées : agrégation préalable                        ║
║    quantite_entree = SUM(quantite) GROUP BY reference            ║
║    Évite la duplication des lignes de ventes                     ║
║                                                                  ║
║  ÉTAPE C — Base ← Entrées agrégées (left join sur 'reference')   ║
║    Colonnes parasites supprimée

,prix_unitaire,quantite_entree,stock_actuel,seuil_d_alerte,quantite_vendue
count,900.00,900.00,900.00,900.00,900.00
mean,"1,218,797.35",94.48,183.41,14.54,39.83
std,"935,030.22",85.70,135.36,6.55,22.93
min,"60,892.92",0.00,0.00,3.00,1.00
25%,"509,670.58",20.75,73.00,9.00,20.00
50%,"997,737.94",80.50,168.00,14.00,38.00
75%,"1,557,695.71",150.00,278.00,20.00,60.00
max,"4,231,088.80",485.00,629.00,25.00,80.00



⚠ stock_actuel < 0 : 0 lignes (0.0%)
  → Ruptures de stock confirmées (anomalie métier déjà identifiée en notebook 01)
  → À gérer lors de la modélisation : clip(lower=0) ou conservation selon le choix métier


---
## 9. Export de la base d'apprentissage

La base est exportée en CSV avec encodage UTF-8 BOM (`utf-8-sig`) pour assurer
la compatibilité avec les outils Microsoft (Excel, Power BI) lors de l'ouverture.

Ce fichier servira de point de départ pour les notebooks suivants :
- `03_analyse_exploratoire.ipynb` — statistiques et visualisations
- `04_modelisation_ml.ipynb` — régression linéaire et KNN

In [12]:
# Export de la base d'apprentissage finale au format CSV
# encoding='utf-8-sig' : ajoute le BOM UTF-8 pour compatibilité Excel/Windows
# index=False : on n'exporte pas l'index pandas (non pertinent comme variable)
SORTIE = 'base_apprentissage_ml.csv'
df_ml.to_csv(SORTIE, index=False, encoding='utf-8-sig')

print(f'✓ Base exportée → {SORTIE}')
print(f'  Taille  : {df_ml.shape[0]} lignes × {df_ml.shape[1]} colonnes')
print(f'  Colonnes: {list(df_ml.columns)}')

✓ Base exportée → base_apprentissage_ml.csv
  Taille  : 900 lignes × 9 colonnes
  Colonnes: ['date', 'reference', 'designation', 'prix_unitaire', 'categorie', 'quantite_entree', 'stock_actuel', 'seuil_d_alerte', 'quantite_vendue']
